# HumanAI Detect - Kisa-Metin Pilot Verisi (Colab)

**Amac:** Model `min_tokens=30` sinirinin ALTINI hic gormedi (bkz. proje notlari,
2026-07-16) -- bu yuzden gercekten kisa metinlerde (kullanicilarin gercek dunyada
girebilecegi ~10-30 kelimelik paragraflar gibi) her zaman "human" tahmin ediyor;
kalibrasyon bile bunu duzeltmiyor (kisa metinde OOD/asiri-guven sorunu, olcumle
dogrulandi). Bu notebook, ana 3000/sinif veri setine DOKUNMADAN, 5-30 kelime
araliginda kucuk (varsayilan 300/sinif) bir "short pilot" havuzu uretir:

- `human_short` : **ZATEN YEREL MAKINEDE URETILDI** (mevcut veriyi yeniden bolme,
  GPU gerekmez) -- bu notebook'ta tekrar uretmene GEREK YOK.
- `ai_raw_short` : Qwen2.5-7B-Instruct ile KISA hedefli YENI uretim -- BU NOTEBOOK, GPU gerekir.
- `ai_humanized_short` : kisa ai_raw'i back-translation ile humanize eder -- BU NOTEBOOK.

**ONEMLI -- once kontrol et:** Bu notebook'u calistirmadan once, yerel makinede
yapilan TUM kod degisikliklerinin (`scripts/collect_short_pilot.py`,
`llm_generators.py` parametrelendirmesi, `configs/data_sources.yaml` -> `short_pilot`
blogu) GitHub'a commit+push edilmis oldugundan emin ol. Colab `git clone` ile
GitHub'daki kodu ceker; yerel degisiklik push edilmezse ESKI kod calisir (bu projede
daha once TAM OLARAK bu yuzden bir tam GPU calistirmasi bosa gitmisti, bkz. proje
notlari 2026-07-05).

In [ ]:
# 1. GPU kontrolu
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# 2. Repo klonla
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
!git clone {GITHUB_REPO} /content/humanai_detect
%cd /content/humanai_detect

In [ ]:
# 3. Bagimliliklari kur
!pip install -e '.[dev,colab]' -q
!pip install bitsandbytes accelerate -q

In [ ]:
# 4. .env olustur (API key gerekmez, transformers/backtranslate tamamen lokal calisir)
env_content = """
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
GEMINI_API_KEY=
LLAMA_API_KEY=
LLAMA_ENDPOINT=
""".strip()
with open('.env', 'w') as f:
    f.write(env_content)
print('.env olusturuldu')

In [ ]:
# 5. (Opsiyonel) short_pilot hedef ornek sayisini degistir -- varsayilan 300/sinif, 5-30 kelime.
# Uzunluk parametrelerini (target_len_mean/std/min/max) degistirmek istersen
# configs/data_sources.yaml -> short_pilot blogunu dogrudan duzenleyebilirsin.
import re

TARGET_COUNT = 300  # <- istersen degistir

with open('configs/data_sources.yaml', 'r', encoding='utf-8') as f:
    content = f.read()

content = re.sub(
    r'(short_pilot:
(?:.*
)*?\s*target_count_per_class:\s*)\d+',
    rf'\g<1>{TARGET_COUNT}',
    content,
)
with open('configs/data_sources.yaml', 'w', encoding='utf-8') as f:
    f.write(content)

print(f'target_count_per_class -> {TARGET_COUNT}')

In [ ]:
# 6. ai_raw_short uret (Qwen2.5-7B-Instruct, KISA hedef: ort. 15 kelime, 5-30 araligi).
# Ana veri setindeki ~1750 kelimelik hedeften COK farkli; max_new_tokens de kisa (60)
# tutuldugu icin bu adim ana 3000/sinif uretiminden (saatler surer) COK daha hizlidir
# (300 kisa ornek icin tahminen birkac dakika).
!python scripts/collect_short_pilot.py --label ai_raw

In [ ]:
# 7. GUVENLIK: ai_raw_short bitince hemen indir (adim 9 -- ai_humanized_short -- daha
# uzun surebilir, oturum kopma riskine karsi onceden indirmek onemli).
from pathlib import Path
from google.colab import files

src = Path('data/raw/ai_raw_short/ai_raw_short.jsonl')
if src.exists():
    n = sum(1 for _ in open(src, encoding='utf-8'))
    print(f'{n} ornek uretildi, indiriliyor...')
    files.download(str(src))
else:
    print('UYARI: dosya bulunamadi.')

**Eger `ai_humanized_short` adimi (asagida) sirasinda oturum koptu/coktu ve yeniden
klonlaman gerektiyse:** `ai_raw_short` uretimini BASTAN YAPMA. Onceki hucreden
indirdigin `ai_raw_short.jsonl`'i asagidaki hucreyle geri yukle, sonra bir sonraki
adima gec.

In [ ]:
# 8. (SADECE oturum koptuysa calistir) ai_raw_short.jsonl'i geri yukle
from pathlib import Path
from google.colab import files

Path('data/raw/ai_raw_short').mkdir(parents=True, exist_ok=True)
uploaded = files.upload()  # ai_raw_short.jsonl'i sec
for name in uploaded:
    Path(name).rename('data/raw/ai_raw_short/ai_raw_short.jsonl')
print('yuklendi.')

In [ ]:
# 9. ai_humanized_short uret (back-translation -- Helsinki-NLP OPUS-MT, Qwen'den
# COK daha kucuk/hizli bir ceviri modeli; GPU'da da CPU'da da calisir ama Colab'da
# GPU varken daha hizlidir).
!python scripts/collect_short_pilot.py --label ai_humanized

In [ ]:
# 10. Sonuclari dogrudan Colab uzerinden yerel makineye indir
from pathlib import Path
from google.colab import files

for label in ['ai_raw_short', 'ai_humanized_short']:
    src = Path(f'data/raw/{label}/{label}.jsonl')
    if src.exists():
        n = sum(1 for _ in open(src, encoding='utf-8'))
        print(f'[{label}] {n} ornek -> indiriliyor')
        files.download(str(src))
    else:
        print(f'[{label}] UYARI: dosya bulunamadi.')

## Indirilen Dosyalari Yerlestirme

Onceki hucreler `ai_raw_short.jsonl` ve `ai_humanized_short.jsonl`'i tarayicinin
varsayilan indirme klasorune (`Downloads`) indirir.

Colab bittikten sonra, kendi makinende:
1. Indirilen `ai_raw_short.jsonl`'i `data/raw/ai_raw_short/ai_raw_short.jsonl`'e tasi.
2. Indirilen `ai_humanized_short.jsonl`'i `data/raw/ai_humanized_short/ai_humanized_short.jsonl`'e tasi.
3. `human_short.jsonl` zaten yerel makinende hazir (`data/raw/human_short/human_short.jsonl`).
4. Bana haber ver -- kalan pipeline adimlarini (preprocess -> build_features -> ana
   veriyle birlestirip yeniden egitim ve OOD tanisi) devam ettiririm.